In [1]:
import socket
import json
import random
import itertools
import math
import heapq
from collections import deque


DIRS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}

MOVE_ORDER = ("UP", "LEFT", "DOWN", "RIGHT")
OPPOSITE = {
    "UP": "DOWN",
    "DOWN": "UP",
    "LEFT": "RIGHT",
    "RIGHT": "LEFT",
}

CELL = 30.0
PACMAN_SPEED = 2.0
NORMAL_GHOST_SPEED = 1.5
FRIGHTENED_GHOST_SPEED = 0.8
EATEN_GHOST_SPEED = 3.0
COLLISION_RADIUS = CELL * 0.75
POWER_FRAMES = 300


class PacmanPlanner:
    """Stateful survival-first planner for the supplied Processing game."""

    def __init__(self):
        self.rows = 0
        self.cols = 0
        self.wall_signature = None
        self.distance_cache = {}
        self.dead_end_depth = {}
        self.reserve_dot = None

        self.mode = "CLEAR"
        self.target = None
        self.target_pellet = None
        self.hunt_target = None
        self.patrol_index = 0
        self.herd_phase = "ACQUIRE"
        self.herd_ghost = None
        self.herd_phase_frames = 0
        self.herd_pellet = None

        self.previous_lives = None
        self.previous_powered = False
        self.previous_pellets = None
        self.previous_score = None
        self.pellets_used = 0
        self.capture_count = 0
        self.recent_pixels = deque(maxlen=90)

    # ------------------------------------------------------------------
    # Public decision entry point
    # ------------------------------------------------------------------

    def choose(self, state):
        maze = state.get("maze") or []
        pac = state.get("pacman") or {}
        ghosts = state.get("ghosts") or []
        if not maze or not maze[0]:
            return "UP"

        self._prepare_map(maze)
        self._observe_state(state)

        pac_cell = self._object_cell(pac)
        pac_pixel = self._object_pixel(pac)
        if not self._walkable(pac_cell, maze):
            return self._heading(pac) or "UP"

        self.recent_pixels.append(
            (round(pac_pixel[0], 1), round(pac_pixel[1], 1))
        )

        power_pellets = self._find_cells(maze, 2)
        dots = self._find_cells(maze, 0)
        edible = [
            g for g in ghosts
            if g.get("frightened", False) and not g.get("eaten", False)
        ]
        dangerous = [
            g for g in ghosts
            if not g.get("frightened", False) and not g.get("eaten", False)
        ]

        mode, target = self._select_mode_and_target(
            state,
            pac_cell,
            power_pellets,
            dots,
            edible,
            dangerous,
            maze,
        )
        self.mode = mode
        self.target = target

        blocked = set()
        # Never burn the next pellet while finishing the current frightened
        # chain. ARM is the only mode allowed to enter a pellet cell.
        if mode != "ARM":
            blocked.update(power_pellets)
        if self.reserve_dot is not None and mode != "ENDGAME":
            blocked.add(self.reserve_dot)

        desired = self._desired_direction(
            pac_cell,
            target,
            maze,
            ghosts,
            blocked,
            mode,
        )

        return self._choose_pixel_safe_action(
            state,
            desired,
            target,
            blocked,
            mode,
        )

    # ------------------------------------------------------------------
    # State and map bookkeeping
    # ------------------------------------------------------------------

    def _prepare_map(self, maze):
        signature = tuple(
            tuple(value == 1 for value in row)
            for row in maze
        )
        if signature == self.wall_signature:
            return

        self.rows = len(maze)
        self.cols = len(maze[0])
        self.wall_signature = signature
        self.distance_cache.clear()
        self.dead_end_depth = self._build_dead_end_depth(maze)

        initial_dots = self._find_cells(maze, 0)
        dead_dots = [
            cell for cell in initial_dots
            if self.dead_end_depth.get(cell, 0) > 0
        ]
        if dead_dots:
            # Keep a remote dead-end dot so the game cannot end before the
            # fourth pellet's ghost hunt is complete.
            pellets = self._find_cells(maze, 2)
            self.reserve_dot = max(
                dead_dots,
                key=lambda cell: (
                    min(
                        (self._distance(cell, p, maze) for p in pellets),
                        default=0,
                    ),
                    self.dead_end_depth.get(cell, 0),
                    cell,
                ),
            )

    def _observe_state(self, state):
        lives = state.get("lives", 3)
        powered = bool(state.get("powered", False))
        score = state.get("score", 0)
        pellets = len(self._find_cells(state["maze"], 2))

        if self.previous_lives is not None and lives < self.previous_lives:
            self._reset_after_life_loss()

        if self.previous_pellets is not None and pellets < self.previous_pellets:
            self.pellets_used += self.previous_pellets - pellets
            self.hunt_target = None

        if self.previous_score is not None:
            delta = score - self.previous_score
            # Score packets arrive after collision checks. A 200-point component
            # therefore identifies one or more captures even when dots are eaten
            # in the same frame.
            if delta >= 200:
                self.capture_count += delta // 200

        if self.previous_powered and not powered:
            self.hunt_target = None

        self.previous_lives = lives
        self.previous_powered = powered
        self.previous_pellets = pellets
        self.previous_score = score

    def _reset_after_life_loss(self):
        self.mode = "CLEAR"
        self.target = None
        self.target_pellet = None
        self.hunt_target = None
        self.patrol_index = 0
        self.herd_phase = "ACQUIRE"
        self.herd_ghost = None
        self.herd_phase_frames = 0
        self.herd_pellet = None
        self.recent_pixels.clear()

    # ------------------------------------------------------------------
    # High-level behavior modes
    # ------------------------------------------------------------------

    def _select_mode_and_target(
        self,
        state,
        pac_cell,
        power_pellets,
        dots,
        edible,
        dangerous,
        maze,
    ):
        powered = bool(state.get("powered", False))

        if powered and edible:
            target = self._choose_frightened_intercept(
                state, pac_cell, edible, maze
            )
            return "HUNT", target

        ordinary_targets = set(dots)
        if self.reserve_dot in ordinary_targets:
            ordinary_targets.remove(self.reserve_dot)

        if power_pellets:
            ready = self._ready_power_pellets(
                state, pac_cell, power_pellets, maze
            )
            if ready:
                pellet = ready[0][2]
                self.target_pellet = pellet
                return "ARM", pellet

            # Clear most of the board before committing to a long bait cycle,
            # but switch to herding early enough that four ghosts can gather.
            if ordinary_targets and len(ordinary_targets) > 28:
                target = self._safest_food_target(
                    pac_cell,
                    ordinary_targets,
                    dangerous,
                    maze,
                )
                if target is not None:
                    return "CLEAR", target

            pellet = self._choose_herd_pellet(
                pac_cell, power_pellets, state["ghosts"], maze
            )
            self.target_pellet = pellet
            target = self._choose_active_herd_target(
                state, pac_cell, pellet, maze
            )
            return "HERD", target

        # No pellets remain. Finish any still-active frightened hunt first.
        if edible:
            target = self._choose_frightened_intercept(
                state, pac_cell, edible, maze
            )
            return "HUNT", target

        if ordinary_targets:
            target = self._safest_food_target(
                pac_cell,
                ordinary_targets,
                dangerous,
                maze,
            )
            return "CLEAR", target

        if self.reserve_dot is not None and self.reserve_dot in dots:
            return "ENDGAME", self.reserve_dot

        return "ENDGAME", self._nearest_cell(pac_cell, dots, maze)

    def _ready_power_pellets(self, state, pac_cell, pellets, maze):
        ghosts = state.get("ghosts") or []
        if len(ghosts) < 4:
            return []

        # Never start a frightened chain while any ghost is still in the
        # central spawn/respawn chamber. A returning eyes ghost becomes normal
        # and lethal immediately on reaching this area, which can turn a hunt
        # through the middle into an unavoidable collision.
        if any(
            self._in_respawn_chamber(self._object_cell(ghost))
            for ghost in ghosts
        ):
            return []

        results = []
        other_pellets = set(pellets)
        for pellet in pellets:
            approach = self._distance(pac_cell, pellet, maze)
            if approach is None:
                continue

            chain_frames, uncertainty = self._estimated_capture_chain_frames(
                pellet, ghosts, maze
            )

            eaten_count = sum(g.get("eaten", False) for g in ghosts)
            lives = state.get("lives", 3)
            reserve = {3: 15, 2: 25}.get(lives, 45)

            # A pellet revives eaten eyes at their current positions. Once all
            # four ghosts from the previous hunt are returning, taking the next
            # pellet is usually the strongest scoring route.
            resurrection_ready = (
                eaten_count == 4
                and chain_frames + uncertainty <= POWER_FRAMES - reserve
            )

            normal_ready = (
                chain_frames + uncertainty <= POWER_FRAMES - reserve
            )

            if lives <= 1 and uncertainty > 20:
                normal_ready = False
                resurrection_ready = False

            if resurrection_ready or normal_ready:
                # Avoid crossing another pellet on the approach.
                path = self._shortest_path(
                    pac_cell,
                    pellet,
                    maze,
                    blocked=other_pellets - {pellet},
                )
                if path:
                    results.append(
                        (approach, chain_frames + uncertainty, pellet)
                    )

        results.sort()
        return results

    def _estimated_capture_chain_frames(self, start, ghosts, maze):
        envelopes = [
            self._frightened_reachable_envelope(g, maze)
            for g in ghosts
        ]
        best = float("inf")
        best_uncertainty = float("inf")

        for order in itertools.permutations(range(len(ghosts))):
            current = start
            frames = 0.0
            uncertainty = 0.0
            possible = True
            for index in order:
                intercept = self._best_envelope_intercept(
                    current,
                    frames,
                    envelopes[index],
                    maze,
                )
                if intercept is None:
                    possible = False
                    break

                ghost = ghosts[index]
                if ghost.get("eaten", False):
                    uncertainty += 2.0
                else:
                    uncertainty += intercept[2]

                frames = intercept[0]
                current = intercept[1]

            if possible and (
                frames < best
                or (frames == best and uncertainty < best_uncertainty)
            ):
                best = min(best, frames)
                best_uncertainty = uncertainty

        return best, best_uncertainty

    def _frightened_reachable_envelope(self, ghost, maze):
        start = self._object_cell(ghost)
        heading = self._heading(ghost)
        if ghost.get("eaten", False):
            return [(0.0, start, 0.0)]

        # The first few frightened frames still follow the current heading.
        result = [(0.0, start, 4.0)]
        current = start
        if heading is not None:
            for step in range(1, 3):
                nxt = dict(self._neighbors(current, maze)).get(heading)
                if nxt is None:
                    break
                result.append(
                    (step * CELL / FRIGHTENED_GHOST_SPEED, nxt, 1.0)
                )
                current = nxt

        # Afterwards frightened turns are random. Represent the moving ghost as
        # a reachable cell envelope rather than assuming it stays in place.
        distances = self._distance_map_from(start, maze, max_depth=7)
        for cell, steps in distances.items():
            if steps == 0:
                continue
            ghost_frames = steps * CELL / FRIGHTENED_GHOST_SPEED
            uncertainty = min(6.0, 1.0 + steps * 0.7)
            result.append((ghost_frames, cell, uncertainty))
        return result

    def _best_envelope_intercept(
        self,
        pac_start,
        elapsed_frames,
        envelope,
        maze,
    ):
        best = None
        for ghost_frames, cell, uncertainty in envelope:
            distance = self._distance(pac_start, cell, maze)
            if distance is None:
                continue
            travel = max(
                0.0,
                distance * CELL / PACMAN_SPEED
                - COLLISION_RADIUS / PACMAN_SPEED,
            )
            arrival = elapsed_frames + travel
            wait = max(0.0, ghost_frames - arrival)
            finish = arrival + min(wait, 18.0)
            timing_error = abs(ghost_frames - arrival)
            candidate = (
                finish + timing_error * 0.08,
                cell,
                uncertainty + min(8.0, timing_error * 0.04),
            )
            if best is None or candidate < best:
                best = candidate
        return best

    def _choose_herd_pellet(self, pac_cell, pellets, ghosts, maze):
        best = None
        for pellet in pellets:
            distances = [
                self._distance(pellet, self._object_cell(g), maze)
                for g in ghosts
                if not g.get("eaten", False)
            ]
            if not distances or any(d is None for d in distances):
                continue

            pac_distance = self._distance(pac_cell, pellet, maze) or 999
            score = (
                sum(distances)
                + max(distances) * 2.5
                + pac_distance * 0.35
            )
            candidate = (score, pellet)
            if best is None or candidate < best:
                best = candidate

        return best[1] if best else min(pellets)

    def _choose_active_herd_target(self, state, pac_cell, pellet, maze):
        ghosts = [
            (index, ghost)
            for index, ghost in enumerate(state["ghosts"])
            if not ghost.get("eaten", False)
            and not ghost.get("frightened", False)
        ]
        if not ghosts:
            return pac_cell

        if pellet != self.herd_pellet:
            self.herd_pellet = pellet
            self.herd_phase = "ACQUIRE"
            self.herd_ghost = None
            self.herd_phase_frames = 0

        self.herd_phase_frames += 1
        distances = {
            index: self._distance(
                self._object_cell(ghost), pellet, maze
            )
            for index, ghost in ghosts
        }
        distances = {
            index: distance
            for index, distance in distances.items()
            if distance is not None
        }
        if not distances:
            return self._choose_herd_patrol_target(
                pac_cell, pellet, state["ghosts"], maze
            )

        farthest = max(distances, key=lambda index: distances[index])
        clustered = all(distance <= 6 for distance in distances.values())

        if clustered:
            self.herd_phase = "GATHER"
            self.herd_ghost = farthest
            self.herd_phase_frames = 0
        elif (
            self.herd_ghost not in distances
            or distances.get(self.herd_ghost, 0) <= 4
        ):
            self.herd_phase = "ACQUIRE"
            self.herd_ghost = farthest
            self.herd_phase_frames = 0

        ghost_lookup = dict(ghosts)
        selected = ghost_lookup[self.herd_ghost]
        selected_cell = self._object_cell(selected)
        pac_to_ghost = self._distance(pac_cell, selected_cell, maze)
        selected_to_pellet = distances[self.herd_ghost]

        if self.herd_phase == "ACQUIRE":
            # Once the straggler can see/chase Pac-Man at useful range, retreat
            # toward the pellet instead of orbiting beside the ghost.
            useful_contact = (
                pac_to_ghost is not None
                and pac_to_ghost <= 6
                and (
                    not self._in_ghost_house(selected_cell)
                    or self._ghost_heading_toward_pacman(
                        selected, state["pacman"]
                    )
                )
            )
            if useful_contact or self.herd_phase_frames > 210:
                self.herd_phase = "LEAD"
                self.herd_phase_frames = 0
            else:
                return self._choose_acquire_point(
                    pac_cell,
                    selected_cell,
                    pellet,
                    state["ghosts"],
                    maze,
                )

        if self.herd_phase == "LEAD":
            if selected_to_pellet <= 4:
                remaining = {
                    index: distance
                    for index, distance in distances.items()
                    if index != self.herd_ghost and distance > 6
                }
                if remaining:
                    self.herd_ghost = max(
                        remaining,
                        key=lambda index: remaining[index],
                    )
                    self.herd_phase = "ACQUIRE"
                    self.herd_phase_frames = 0
                    next_cell = self._object_cell(
                        ghost_lookup[self.herd_ghost]
                    )
                    return self._choose_acquire_point(
                        pac_cell,
                        next_cell,
                        pellet,
                        state["ghosts"],
                        maze,
                    )
                self.herd_phase = "GATHER"
                self.herd_phase_frames = 0
            else:
                target = self._choose_lead_point(
                    pac_cell,
                    selected_cell,
                    pellet,
                    state["ghosts"],
                    maze,
                )
                if target is not None:
                    return target
                # If the bait corridor becomes unsafe, fall back to a spacious
                # staging loop rather than forcing contact.
                self.herd_phase = "GATHER"
                self.herd_phase_frames = 0

        # GATHER: circle outside the pellet while pulling the remaining group
        # through the staging area. A newly distant straggler restarts Acquire.
        farthest = max(distances, key=lambda index: distances[index])
        if distances[farthest] > 8 and self.herd_phase_frames > 45:
            self.herd_ghost = farthest
            self.herd_phase = "ACQUIRE"
            self.herd_phase_frames = 0
            return self._choose_acquire_point(
                pac_cell,
                self._object_cell(ghost_lookup[farthest]),
                pellet,
                state["ghosts"],
                maze,
            )

        return self._choose_gather_point(
            pac_cell,
            pellet,
            state["ghosts"],
            maze,
        )

    def _choose_acquire_point(
        self,
        pac_cell,
        ghost_cell,
        pellet,
        ghosts,
        maze,
    ):
        ghost_to_pellet = self._distance(ghost_cell, pellet, maze) or 999
        candidates = []
        for cell in self._open_cells(maze):
            to_ghost = self._distance(cell, ghost_cell, maze)
            to_pellet = self._distance(cell, pellet, maze)
            if to_ghost is None or to_pellet is None:
                continue
            if not 3 <= to_ghost <= 6:
                continue
            if to_pellet >= ghost_to_pellet:
                continue
            if self.dead_end_depth.get(cell, 0) > 0:
                continue
            if self._degree(cell, maze) < 2:
                continue

            route = self._distance(pac_cell, cell, maze)
            if route is None:
                continue
            clearance = self._normal_ghost_clearance(cell, ghosts, maze)
            score = (
                to_ghost * 16
                + route * 3
                + to_pellet * 1.5
                - min(clearance, 8) * 8
            )
            candidates.append((score, cell))

        if candidates:
            candidates.sort()
            return candidates[0][1]
        return self._choose_herd_patrol_target(
            pac_cell, pellet, ghosts, maze
        )

    def _choose_lead_point(
        self,
        pac_cell,
        ghost_cell,
        pellet,
        ghosts,
        maze,
    ):
        ghost_distance = self._distance(ghost_cell, pellet, maze)
        if ghost_distance is None:
            return None

        candidates = []
        for cell in self._open_cells(maze):
            to_pellet = self._distance(cell, pellet, maze)
            to_ghost = self._distance(cell, ghost_cell, maze)
            if to_pellet is None or to_ghost is None:
                continue
            if not 2 <= to_pellet <= max(4, ghost_distance - 1):
                continue
            if not 2 <= to_ghost <= 6:
                continue
            if self.dead_end_depth.get(cell, 0) > 0:
                continue
            if self._degree(cell, maze) < 2:
                continue

            route = self._distance(pac_cell, cell, maze)
            if route is None:
                continue
            clearance = self._normal_ghost_clearance(cell, ghosts, maze)
            # Remain ahead of the selected ghost while making concrete progress
            # toward the pellet. Clearance prevents a brave little suicide march.
            score = (
                to_pellet * 22
                + abs(to_ghost - 4) * 18
                + route * 2
                - min(clearance, 8) * 9
            )
            candidates.append((score, cell))

        if not candidates:
            return None
        candidates.sort()
        return candidates[0][1]

    def _choose_gather_point(self, pac_cell, pellet, ghosts, maze):
        recent_cells = {
            self._pixel_cell(pixel)
            for pixel in list(self.recent_pixels)[-55:]
        }
        candidates = []
        for cell in self._open_cells(maze):
            to_pellet = self._distance(cell, pellet, maze)
            if to_pellet is None or not 2 <= to_pellet <= 4:
                continue
            if self.dead_end_depth.get(cell, 0) > 0:
                continue
            if self._degree(cell, maze) < 2:
                continue

            route = self._distance(pac_cell, cell, maze)
            if route is None:
                continue
            ghost_distances = [
                self._distance(cell, self._object_cell(ghost), maze)
                for ghost in ghosts
                if not ghost.get("eaten", False)
                and not ghost.get("frightened", False)
            ]
            ghost_distances = [
                distance for distance in ghost_distances
                if distance is not None
            ]
            pull = sum(max(0, 9 - distance) for distance in ghost_distances)
            nearest = min(ghost_distances, default=9)
            revisit = 70 if cell in recent_cells else 0
            score = (
                abs(to_pellet - 3) * 25
                + route * 3
                + max(0, 3 - nearest) * 120
                - pull * 7
                + revisit
            )
            candidates.append((score, cell))

        if not candidates:
            return pac_cell
        candidates.sort()
        return candidates[0][1]

    def _normal_ghost_clearance(self, cell, ghosts, maze):
        distances = [
            self._distance(cell, self._object_cell(ghost), maze)
            for ghost in ghosts
            if not ghost.get("eaten", False)
            and not ghost.get("frightened", False)
        ]
        distances = [distance for distance in distances if distance is not None]
        return min(distances, default=20)

    def _ghost_heading_toward_pacman(self, ghost, pac):
        gx, gy = self._object_pixel(ghost)
        px, py = self._object_pixel(pac)
        dx = ghost.get("dx", 0)
        dy = ghost.get("dy", 0)
        current = self._wrapped_distance((gx, gy), (px, py))
        future = self._wrapped_distance(
            (self._wrap_x(gx + dx * 6), gy + dy * 6),
            (px, py),
        )
        return future < current

    def _choose_herd_patrol_target(self, pac_cell, pellet, ghosts, maze):
        if pellet is None:
            return pac_cell

        ring = []
        for cell in self._open_cells(maze):
            distance = self._distance(cell, pellet, maze)
            if distance is None or not 2 <= distance <= 5:
                continue
            if self.dead_end_depth.get(cell, 0) > 0:
                continue
            if self._degree(cell, maze) < 2:
                continue
            ring.append(cell)

        if not ring:
            return pac_cell

        ghost_cells = [
            self._object_cell(g)
            for g in ghosts
            if not g.get("eaten", False)
        ]
        recent_cells = {
            self._pixel_cell(pixel)
            for pixel in list(self.recent_pixels)[-40:]
        }

        def score(cell):
            ghost_distance = min(
                (
                    self._distance(cell, ghost, maze)
                    for ghost in ghost_cells
                    if self._distance(cell, ghost, maze) is not None
                ),
                default=20,
            )
            approach = self._distance(pac_cell, cell, maze) or 999
            pull = sum(
                max(0, 9 - (self._distance(cell, ghost, maze) or 20))
                for ghost in ghost_cells
            )
            revisit = 80 if cell in recent_cells else 0
            return (
                min(ghost_distance, 8) * 30
                + pull * 5
                + self._degree(cell, maze) * 25
                - approach * 4
                - revisit
            )

        return max(ring, key=lambda cell: (score(cell), cell))

    def _safest_food_target(self, start, targets, dangerous, maze):
        if not targets:
            return None

        ghost_cells = [self._object_cell(g) for g in dangerous]
        best = None
        for target in targets:
            distance = self._distance(start, target, maze)
            if distance is None:
                continue
            ghost_clearance = min(
                (
                    self._distance(target, ghost, maze)
                    for ghost in ghost_cells
                    if self._distance(target, ghost, maze) is not None
                ),
                default=20,
            )
            dead_depth = self.dead_end_depth.get(target, 0)
            value = (
                distance
                + dead_depth * 3
                - min(ghost_clearance, 8) * 0.7
            )
            candidate = (value, target)
            if best is None or candidate < best:
                best = candidate
        return best[1] if best else None

    # ------------------------------------------------------------------
    # Frightened ghost interception
    # ------------------------------------------------------------------

    def _choose_frightened_intercept(self, state, pac_cell, edible, maze):
        timer = max(0, state.get("powerTimer", 0))
        candidates = []

        for index, ghost in enumerate(edible):
            ghost_cell = self._object_cell(ghost)
            predicted = self._predict_ghost_cells(
                ghost,
                state["pacman"],
                maze,
                horizon=105,
                frightened=True,
            )

            best_intercept = None
            for frame, cell in predicted:
                route = self._distance(pac_cell, cell, maze)
                if route is None:
                    continue
                pac_frames = max(
                    0.0,
                    route * CELL / PACMAN_SPEED
                    - COLLISION_RADIUS / PACMAN_SPEED,
                )
                slack = frame - pac_frames
                timer_slack = timer - pac_frames
                if timer_slack < 8:
                    continue

                # Prefer an interception that Pac-Man can reach just before the
                # ghost, rather than following the ghost's stale current cell.
                score = (
                    abs(slack) * 1.8
                    + pac_frames * 0.08
                    - min(timer_slack, 90) * 0.03
                    + self._respawn_intercept_penalty(
                        cell,
                        pac_frames,
                        state["ghosts"],
                        maze,
                    )
                )
                item = (score, pac_frames, cell)
                if best_intercept is None or item < best_intercept:
                    best_intercept = item

            if best_intercept is None:
                route = self._distance(pac_cell, ghost_cell, maze)
                if route is None:
                    continue
                pac_frames = route * CELL / PACMAN_SPEED
                best_intercept = (pac_frames, pac_frames, ghost_cell)

            sticky = 0
            ghost_id = self._ghost_identity(ghost, index)
            if self.hunt_target == ghost_id:
                sticky = -28

            candidates.append(
                (
                    best_intercept[0] + sticky,
                    best_intercept[1],
                    ghost_id,
                    best_intercept[2],
                )
            )

        if not candidates:
            return self._object_cell(edible[0])

        candidates.sort()
        chosen = candidates[0]
        self.hunt_target = chosen[2]
        return chosen[3]

    def _respawn_intercept_penalty(
        self,
        intercept_cell,
        pac_frames,
        ghosts,
        maze,
    ):
        penalty = 0.0
        for ghost_index, ghost in enumerate(ghosts):
            if not ghost.get("eaten", False):
                continue
            eta = self._eaten_respawn_eta(ghost, maze)
            if eta is None:
                continue
            spawn_cell = self._pixel_cell(
                self._ghost_spawn_pixel(ghost_index)
            )
            distance = self._distance(
                intercept_cell, spawn_cell, maze
            )
            if distance is None or distance > 4:
                continue
            timing_gap = abs(pac_frames - eta)
            if timing_gap <= 90:
                penalty += (
                    (5 - distance) * 65.0
                    + (90 - timing_gap) * 2.0
                )
        return penalty

    def _predict_ghost_cells(
        self,
        ghost,
        pac,
        maze,
        horizon,
        frightened,
    ):
        x, y = self._object_pixel(ghost)
        dx = float(ghost.get("dx", 0))
        dy = float(ghost.get("dy", 0))
        speed = FRIGHTENED_GHOST_SPEED if frightened else NORMAL_GHOST_SPEED
        predictions = [(0, self._pixel_cell((x, y)))]

        for frame in range(1, horizon + 1):
            nx = self._wrap_x(x + dx * speed)
            ny = y + dy * speed
            if self._pixel_walkable(nx, ny, maze):
                x, y = nx, ny
            else:
                direction = self._ghost_turn(
                    (x, y),
                    (dx, dy),
                    pac,
                    maze,
                    frightened,
                )
                dr, dc = DIRS[direction]
                dx, dy = float(dc), float(dr)

            if frame % 15 == 0:
                predictions.append((frame, self._pixel_cell((x, y))))

        return predictions

    # ------------------------------------------------------------------
    # Cell route planning
    # ------------------------------------------------------------------

    def _desired_direction(
        self,
        start,
        target,
        maze,
        ghosts,
        blocked,
        mode,
    ):
        if target is None or start == target:
            return None

        path = self._weighted_path(
            start,
            target,
            maze,
            ghosts,
            blocked,
            mode,
        )
        if len(path) < 2:
            return None
        return self._direction_between(path[0], path[1])

    def _weighted_path(
        self,
        start,
        target,
        maze,
        ghosts,
        blocked,
        mode,
    ):
        if target is None:
            return []

        queue = [(0.0, 0, start)]
        counter = 1
        cost = {start: 0.0}
        parent = {}

        while queue:
            _, _, cell = heapq.heappop(queue)
            if cell == target:
                break

            for _, nxt in self._neighbors(cell, maze):
                if nxt in blocked and nxt != target:
                    continue

                step = 1.0
                if mode not in ("HUNT", "ARM"):
                    step += self.dead_end_depth.get(nxt, 0) * 1.7

                pixel = self._cell_center(nxt)
                projected_pac_frames = (
                    cost[cell] + step
                ) * CELL / PACMAN_SPEED
                for ghost_index, ghost in enumerate(ghosts):
                    if ghost.get("eaten", False):
                        step += self._respawn_route_penalty(
                            nxt,
                            projected_pac_frames,
                            ghost,
                            ghost_index,
                            maze,
                        )
                        continue
                    if ghost.get("frightened", False):
                        continue
                    distance = self._wrapped_distance(
                        pixel,
                        self._object_pixel(ghost),
                    )
                    if distance < CELL * 1.1:
                        step += 18.0
                    elif distance < CELL * 2.2:
                        step += 5.0

                new_cost = cost[cell] + step
                if new_cost >= cost.get(nxt, float("inf")):
                    continue
                cost[nxt] = new_cost
                parent[nxt] = cell
                heuristic = self._distance(nxt, target, maze) or 0
                heapq.heappush(
                    queue,
                    (new_cost + heuristic * 0.35, counter, nxt),
                )
                counter += 1

        if target not in cost:
            return []

        path = [target]
        while path[-1] != start:
            path.append(parent[path[-1]])
        path.reverse()
        return path

    # ------------------------------------------------------------------
    # Pixel-level action safety and movement
    # ------------------------------------------------------------------

    def _choose_pixel_safe_action(
        self,
        state,
        desired,
        target,
        blocked,
        mode,
    ):
        maze = state["maze"]
        pac = state["pacman"]
        ghosts = state["ghosts"]
        lives = state.get("lives", 3)
        current = self._heading(pac)
        pac_pixel = self._object_pixel(pac)

        actions = [
            action for action in MOVE_ORDER
            if self._command_is_pixel_legal(pac_pixel, action, maze)
        ]
        if not actions:
            return current or "UP"

        evaluations = []
        for action in actions:
            pac_trace = self._simulate_pac_trace(
                pac,
                action,
                maze,
                frames=42,
            )
            risk, collision = self._trace_risk(
                pac_trace,
                ghosts,
                pac,
                maze,
                mode,
            )

            next_pixel = pac_trace[0]
            next_cell = self._pixel_cell(next_pixel)
            pellet_penalty = 0
            if next_cell in blocked:
                pellet_penalty = 100000

            target_progress = 0.0
            if target is not None:
                before = self._distance(
                    self._pixel_cell(pac_pixel), target, maze
                )
                after = self._distance(next_cell, target, maze)
                if before is not None and after is not None:
                    target_progress = (before - after) * 38.0
                target_center = self._cell_center(target)
                before_pixels = self._wrapped_distance(pac_pixel, target_center)
                after_pixels = self._wrapped_distance(next_pixel, target_center)
                target_progress += (before_pixels - after_pixels) * 0.8

            direction_bonus = 0.0
            if action == desired:
                direction_bonus += 75.0
            if action == current:
                direction_bonus += 12.0
            if current and action == OPPOSITE[current]:
                direction_bonus -= 18.0

            wall_lane_bonus = self._wall_lane_safety_bonus(
                next_pixel, ghosts, maze, mode
            )
            loop_penalty = self._recent_pixel_penalty(next_pixel)

            hard_reject = collision
            if lives <= 1 and risk > 15:
                hard_reject = True

            # With spare lives, an expiring frightened capture may use a narrow
            # margin, but a predicted actual collision is never intentional.
            risk_weight = 9.0 if lives <= 1 else 5.0
            if mode == "HUNT" and state.get("powerTimer", 0) < 75:
                risk_weight *= 0.72

            score = (
                target_progress
                + direction_bonus
                + wall_lane_bonus
                - risk * risk_weight
                - loop_penalty
                - pellet_penalty
            )
            evaluations.append((hard_reject, score, action))

        safe = [item for item in evaluations if not item[0]]
        pool = safe if safe else evaluations
        pool.sort(key=lambda item: (item[1], item[2] == current), reverse=True)
        return pool[0][2]

    def _simulate_pac_trace(self, pac, command, maze, frames):
        x, y = self._object_pixel(pac)
        dx = float(pac.get("dx", 0))
        dy = float(pac.get("dy", 0))
        dr, dc = DIRS[command]
        qdx, qdy = float(dc), float(dr)
        trace = []

        for _ in range(frames):
            tx = self._wrap_x(x + qdx * PACMAN_SPEED)
            ty = y + qdy * PACMAN_SPEED
            if self._pixel_walkable(tx, ty, maze):
                dx, dy = qdx, qdy

            nx = self._wrap_x(x + dx * PACMAN_SPEED)
            ny = y + dy * PACMAN_SPEED
            if self._pixel_walkable(nx, ny, maze):
                x, y = nx, ny
            trace.append((x, y))

        return trace

    def _trace_risk(self, pac_trace, ghosts, pac, maze, mode):
        risk = 0.0
        collision = False
        ghost_traces = [
            self._simulate_ghost_trace(
                ghost,
                ghost_index,
                pac_trace,
                pac,
                maze,
                len(pac_trace),
            )
            for ghost_index, ghost in enumerate(ghosts)
        ]

        for ghost, trace in zip(ghosts, ghost_traces):
            for frame, (pac_pixel, ghost_state) in enumerate(
                zip(pac_trace, trace), start=1
            ):
                ghost_pixel, lethal = ghost_state
                distance = self._wrapped_distance(pac_pixel, ghost_pixel)
                if not lethal:
                    edible = (
                        ghost.get("frightened", False)
                        and not ghost.get("eaten", False)
                    )
                    if mode == "HUNT":
                        if edible:
                            risk -= max(
                                0.0,
                                COLLISION_RADIUS + 8 - distance,
                            ) * 0.45
                    continue

                uncertainty = min(10.0, frame * 0.22)
                margin = COLLISION_RADIUS + uncertainty
                if distance < COLLISION_RADIUS:
                    collision = True
                    risk += 500.0
                elif distance < margin + 18:
                    risk += (margin + 18 - distance) * (
                        1.2 - min(frame, 35) / 70.0
                    )

        return risk, collision

    def _simulate_ghost_trace(
        self,
        ghost,
        ghost_index,
        pac_trace,
        pac,
        maze,
        frames,
    ):
        x, y = self._object_pixel(ghost)
        dx = float(ghost.get("dx", 0))
        dy = float(ghost.get("dy", 0))
        frightened = ghost.get("frightened", False)
        eaten = ghost.get("eaten", False)
        speed = (
            EATEN_GHOST_SPEED
            if eaten
            else FRIGHTENED_GHOST_SPEED
            if frightened
            else NORMAL_GHOST_SPEED
        )
        trace = []

        for frame in range(frames):
            nx = self._wrap_x(x + dx * speed)
            ny = y + dy * speed
            if self._pixel_walkable(nx, ny, maze):
                x, y = nx, ny
            else:
                if eaten:
                    pac_target = {
                        "px": 10.5 * CELL,
                        "py": 9.5 * CELL,
                    }
                else:
                    pac_target = {
                        "px": pac_trace[min(frame, len(pac_trace) - 1)][0],
                        "py": pac_trace[min(frame, len(pac_trace) - 1)][1],
                    }
                direction = self._ghost_turn(
                    (x, y),
                    (dx, dy),
                    pac_target,
                    maze,
                    frightened and not eaten,
                )
                dr, dc = DIRS[direction]
                dx, dy = float(dc), float(dr)

            if eaten and self._wrapped_distance(
                (x, y),
                (10.5 * CELL, 9.5 * CELL),
            ) < CELL * 0.6:
                x, y = self._ghost_spawn_pixel(ghost_index)
                dx, dy = -1.0, 0.0
                eaten = False
                frightened = False
                speed = NORMAL_GHOST_SPEED

            trace.append(((x, y), not eaten and not frightened))

        return trace

    def _ghost_spawn_pixel(self, ghost_index):
        starts = (
            (9.5 * CELL, 9.5 * CELL),
            (10.5 * CELL, 9.5 * CELL),
            (11.5 * CELL, 9.5 * CELL),
            (10.5 * CELL, 10.5 * CELL),
        )
        return starts[ghost_index % len(starts)]

    def _eaten_respawn_eta(self, ghost, maze):
        if not ghost.get("eaten", False):
            return None
        start = self._object_cell(ghost)
        home = (9, 10)
        distance = self._distance(start, home, maze)
        if distance is None:
            return None
        pixel_remainder = self._wrapped_distance(
            self._object_pixel(ghost),
            self._cell_center(start),
        )
        return max(
            0.0,
            distance * CELL / EATEN_GHOST_SPEED
            - COLLISION_RADIUS / EATEN_GHOST_SPEED
            - pixel_remainder / EATEN_GHOST_SPEED,
        )

    def _respawn_route_penalty(
        self,
        cell,
        pac_frames,
        ghost,
        ghost_index,
        maze,
    ):
        eta = self._eaten_respawn_eta(ghost, maze)
        if eta is None:
            return 0.0
        spawn_cell = self._pixel_cell(
            self._ghost_spawn_pixel(ghost_index)
        )
        spawn_distance = self._distance(cell, spawn_cell, maze)
        if spawn_distance is None or spawn_distance > 3:
            return 0.0

        timing_gap = abs(pac_frames - eta)
        if timing_gap > 75:
            return 0.0
        return (
            (4 - spawn_distance) * 18.0
            + (75 - timing_gap) * 0.9
        )

    def _ghost_turn(self, pixel, heading, pac, maze, frightened):
        x, y = pixel
        dx, dy = heading
        reverse = (-dx, -dy)
        choices = []

        for action in MOVE_ORDER:
            dr, dc = DIRS[action]
            ndx, ndy = float(dc), float(dr)
            if (ndx, ndy) == reverse:
                continue
            nx = self._wrap_x(x + ndx * NORMAL_GHOST_SPEED * 5)
            ny = y + ndy * NORMAL_GHOST_SPEED * 5
            if not self._pixel_walkable(nx, ny, maze):
                continue

            if frightened:
                # Stable pseudo-random ordering approximates frightened turns
                # without making the planner itself nondeterministic.
                value = (
                    int(nx * 17 + ny * 31 + len(self.recent_pixels) * 13)
                    % 997
                )
            else:
                target = self._object_pixel(pac)
                value = self._wrapped_distance((nx, ny), target)
            choices.append((value, action))

        if not choices:
            for action in MOVE_ORDER:
                dr, dc = DIRS[action]
                nx = self._wrap_x(x + dc * NORMAL_GHOST_SPEED)
                ny = y + dr * NORMAL_GHOST_SPEED
                if self._pixel_walkable(nx, ny, maze):
                    return action
            return "LEFT"

        choices.sort()
        return choices[0][1]

    def _wall_lane_safety_bonus(self, pac_pixel, ghosts, maze, mode):
        if mode == "HUNT":
            return 0.0

        x, y = pac_pixel
        cell = self._pixel_cell(pac_pixel)
        center = self._cell_center(cell)
        bonus = 0.0

        for ghost in ghosts:
            if ghost.get("eaten", False) or ghost.get("frightened", False):
                continue
            gx, gy = self._object_pixel(ghost)

            # Reward occupying the side of a wide corridor farther from a
            # nearby ghost. This uses real pixel separation instead of treating
            # a whole grid row or column as uniformly dangerous.
            if abs(gx - x) < CELL * 2 and abs(gy - y) < CELL * 2:
                if abs(gx - x) > abs(gy - y):
                    bonus += abs(y - center[1]) * 0.35
                else:
                    bonus += abs(x - center[0]) * 0.35

        return bonus

    def _recent_pixel_penalty(self, pixel):
        if not self.recent_pixels:
            return 0.0
        count = sum(
            self._wrapped_distance(pixel, previous) < 4.1
            for previous in self.recent_pixels
        )
        return count * 3.5

    # ------------------------------------------------------------------
    # Graph and geometry helpers
    # ------------------------------------------------------------------

    def _shortest_path(self, start, target, maze, blocked=frozenset()):
        if start == target:
            return [start]
        queue = deque([start])
        parent = {start: None}

        while queue:
            cell = queue.popleft()
            for _, nxt in self._neighbors(cell, maze):
                if nxt in blocked and nxt != target:
                    continue
                if nxt in parent:
                    continue
                parent[nxt] = cell
                if nxt == target:
                    path = [target]
                    while path[-1] != start:
                        path.append(parent[path[-1]])
                    path.reverse()
                    return path
                queue.append(nxt)
        return []

    def _distance(self, start, target, maze):
        if start is None or target is None:
            return None
        key = (start, target)
        if key in self.distance_cache:
            return self.distance_cache[key]
        path = self._shortest_path(start, target, maze)
        result = len(path) - 1 if path else None
        self.distance_cache[key] = result
        self.distance_cache[(target, start)] = result
        return result

    def _distance_map_from(self, start, maze, max_depth=None):
        distances = {start: 0}
        queue = deque([start])
        while queue:
            cell = queue.popleft()
            if max_depth is not None and distances[cell] >= max_depth:
                continue
            for _, nxt in self._neighbors(cell, maze):
                if nxt in distances:
                    continue
                distances[nxt] = distances[cell] + 1
                queue.append(nxt)
        return distances

    def _neighbors(self, cell, maze):
        r, c = cell
        result = []
        for action in MOVE_ORDER:
            dr, dc = DIRS[action]
            nr = r + dr
            nc = (c + dc) % self.cols
            nxt = (nr, nc)
            if self._walkable(nxt, maze):
                result.append((action, nxt))
        return result

    def _build_dead_end_depth(self, maze):
        cells = set(self._open_cells(maze))
        adjacency = {
            cell: {nxt for _, nxt in self._neighbors(cell, maze)}
            for cell in cells
        }
        degree = {cell: len(neighbors) for cell, neighbors in adjacency.items()}
        queue = deque(cell for cell, value in degree.items() if value <= 1)
        removed = set()
        depth = {}

        while queue:
            cell = queue.popleft()
            if cell in removed:
                continue
            removed.add(cell)
            remaining = [n for n in adjacency[cell] if n not in removed]
            depth[cell] = 1 + max(
                (depth.get(n, 0) for n in adjacency[cell]),
                default=0,
            )
            for nxt in remaining:
                degree[nxt] -= 1
                if degree[nxt] == 1:
                    queue.append(nxt)
        return depth

    def _degree(self, cell, maze):
        return len(self._neighbors(cell, maze))

    def _walkable(self, cell, maze):
        if cell is None:
            return False
        r, c = cell
        return (
            isinstance(r, int)
            and isinstance(c, int)
            and 0 <= r < self.rows
            and 0 <= c < self.cols
            and maze[r][c] != 1
        )

    def _pixel_walkable(self, x, y, maze):
        return self._walkable(self._pixel_cell((x, y)), maze)

    def _command_is_pixel_legal(self, pixel, action, maze):
        x, y = pixel
        dr, dc = DIRS[action]
        nx = self._wrap_x(x + dc * PACMAN_SPEED)
        ny = y + dr * PACMAN_SPEED
        return self._pixel_walkable(nx, ny, maze)

    def _open_cells(self, maze):
        for r, row in enumerate(maze):
            for c, value in enumerate(row):
                if value != 1:
                    yield (r, c)

    def _find_cells(self, maze, value):
        return {
            (r, c)
            for r, row in enumerate(maze)
            for c, current in enumerate(row)
            if current == value
        }

    def _nearest_cell(self, start, targets, maze):
        choices = [
            (self._distance(start, target, maze), target)
            for target in targets
        ]
        choices = [item for item in choices if item[0] is not None]
        return min(choices)[1] if choices else None

    def _direction_between(self, first, second):
        for action, (dr, dc) in DIRS.items():
            nr = first[0] + dr
            nc = (first[1] + dc) % self.cols
            if (nr, nc) == second:
                return action
        return None

    def _heading(self, obj):
        dx = obj.get("dx", 0)
        dy = obj.get("dy", 0)
        if dx > 0:
            return "RIGHT"
        if dx < 0:
            return "LEFT"
        if dy > 0:
            return "DOWN"
        if dy < 0:
            return "UP"
        return None

    def _object_cell(self, obj):
        if "grid_r" in obj and "grid_c" in obj:
            return (int(obj["grid_r"]), int(obj["grid_c"]))
        return self._pixel_cell(self._object_pixel(obj))

    def _object_pixel(self, obj):
        if "px" in obj and "py" in obj:
            return (float(obj["px"]), float(obj["py"]))
        cell = self._object_cell(obj)
        return self._cell_center(cell)

    def _pixel_cell(self, pixel):
        x, y = pixel
        c = int(self._wrap_x(x) / CELL)
        r = int(y / CELL)
        return (r, c)

    def _cell_center(self, cell):
        r, c = cell
        return ((c + 0.5) * CELL, (r + 0.5) * CELL)

    def _wrap_x(self, x):
        width = self.cols * CELL
        return x % width

    def _wrapped_distance(self, first, second):
        dx = abs(first[0] - second[0])
        width = self.cols * CELL
        dx = min(dx, width - dx)
        dy = first[1] - second[1]
        return math.hypot(dx, dy)

    def _in_ghost_house(self, cell):
        return self._in_respawn_chamber(cell)

    def _in_respawn_chamber(self, cell):
        r, c = cell
        # Exact central chamber containing all four spawn points and the
        # GHOST_HOME target. Row 6 and the side corridors count as outside.
        return 7 <= r <= 10 and 8 <= c <= 12

    def _ghost_identity(self, ghost, fallback):
        # Ghost array order is stable in the Processing sketch.
        return fallback


_PLANNER = PacmanPlanner()


def get_best_move(state):
    """Return one legal movement command for the current game-state packet."""
    return _PLANNER.choose(state)


print(
    "Survival-first PacmanPlanner loaded: "
    "pixel prediction, herding, interception, and 4,870-point endgame."
)


Survival-first PacmanPlanner loaded: pixel prediction, herding, interception, and 4,870-point endgame.


In [2]:
import time

# --- Connect to the Processing server on 127.0.0.1:5204 ---
s = None
reader = None
writer = None

try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(10)          # 10-second timeout for connection & reads
    # Note: 127.0.0.1 is used instead of 'localhost' to prevent connection errors on Mac
    s.connect(("127.0.0.1", 5204))
    reader = s.makefile("r")  # Separate read buffer
    writer = s.makefile("w")  # Separate write buffer
    print("Successfully connected to Processing Game!")
except (ConnectionRefusedError, OSError, TimeoutError) as e:
    print(f"ERROR: Could not connect — {e}")
    print("Is the Processing sketch running?")
    if s:
        s.close()
    s = None

if s:
    start_sent = False  # Track whether we already sent START for this non-playing state
    try:
        while True:
            # 1. Read the JSON state from Processing
            try:
                line = reader.readline()
            except socket.timeout:
                print("WARNING: No data received (timeout). Retrying...")
                continue

            if not line:
                print("Server closed the connection.")
                break

            # 2. Parse the JSON
            try:
                state = json.loads(line)
            except json.JSONDecodeError:
                continue  # Skip broken packets

            # 3. Decide what to do
            if state["gameState"] != 1:
                if not start_sent:
                    action = "START"
                    start_sent = True
                else:
                    # Already sent START; just keep reading until game resumes
                    action = "START"
            else:
                start_sent = False  # Game is active again, reset the flag
                action = get_best_move(state)

            # 4. Send the command back to Processing
            command = json.dumps({"action": action})
            writer.write(command + "\n")
            writer.flush()

    except KeyboardInterrupt:
        print("\nAI Stopped by User.")
    except Exception as e:
        print(f"Unexpected error: {e}")
    finally:
        reader.close()
        writer.close()
        s.close()
        print("Connection closed.")

Successfully connected to Processing Game!
Server closed the connection.
Connection closed.
